In [ ]:
from finbourne_sdk_utils.jupyter_tools import toggle_code

"""
IBOR User Journey

A day in the life of an IBOR using LUSID

Attributes
----------
instruments
quotes
transaction configuration
sub-holding keys
aggregation
corporate actions
cocoon
aggregation
results store
valuation reconciliation
"""

toggle_code("Toggle Docstring")

### Import Libraries

In [ ]:
from datetime import datetime, timedelta
import copy
import json
import os
import socket

import pandas as pd
import numpy as np
import pytz
import matplotlib.pyplot as plt

from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException

import finbourne.sdk.services.lusid as lu

from finbourne.sdk.services.lusid.api import (
    InstrumentsApi, 
    TransactionPortfoliosApi,
    ConfigurationRecipeApi,
    CorporateActionSourcesApi,
    AggregationApi, 
    PortfoliosApi,
    DerivedTransactionPortfoliosApi,
    ReconciliationsApi,
    StructuredResultDataApi,
    PropertyDefinitionsApi,
    TransactionConfigurationApi,
    TransactionPortfoliosApi
)

from finbourne.sdk.services.lusid.models import (
    CreateTransactionPortfolioRequest, 
    CreateCorporateActionSourceRequest, 
    CreatePropertyDefinitionRequest,
    ResourceId, 
    CreatePortfolioDetails,
    UpsertCorporateActionRequest, 
    CorporateActionTransitionRequest,
    CorporateActionTransitionComponentRequest,
    ConfigurationRecipe,
    PricingContext,
    PortfolioResultDataKeyRule,
    UpsertRecipeRequest,
    CreateDataMapRequest,
    DataMapKey,
    DataMapping,
    DataDefinition,
    MarketContext, 
    MarketDataKeyRule, 
    MarketContextSuppliers, 
    MarketOptions,
    ValuationRequest,
    AggregateSpec, 
    CreateDerivedTransactionPortfolioRequest,
    ValuationsReconciliationRequest,
    StructuredResultData,
    StructuredResultDataId,
    UpsertStructuredResultDataRequest,
    SideDefinitionRequest,
    TransactionTypeAlias, 
    PropertyValue, 
    PerpetualProperty,
    PortfolioEntityId,
    TransactionTypeMovement, 
    TransactionTypeRequest,
    ValuationSchedule,
    SidesDefinitionRequest
)

from finbourne_sdk_utils.cocoon.cocoon import load_from_data_frame
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.cocoon.utilities import create_scope_id

from finbourne_sdk_utils.cocoon.cocoon_printer import (
    format_instruments_response,
    format_quotes_response,
    format_portfolios_response,
    format_transactions_response,
    format_holdings_response
)

### Set Up Top Level Variables

In [ ]:
# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook")

scope = "IBORUserJourney"
ibor_portfolio_code = "IBOR" + "-" + create_scope_id()
ibor_portfolio_creation_date = datetime(2015, 1, 1, tzinfo=pytz.UTC)

# The timezone that you are operating in
us_tz = pytz.timezone('America/New_York')

## Day 1 - 28th of April 2020

## 9am - Start with Accounting Book of Record (ABOR)

Will ensure that you have the following loaded internally to support the start of day ABOR

- Instruments
- Market Data Quotes

Then you load in the start of day ABOR

- Create Portfolio (once off, usually already done)
- Set Holdings

#### Instruments

In [ ]:
# Load and display instruments from CSV file
instruments = pd.read_csv("./data/IBORFlows_Securities.csv")
instruments.head(n=15)

In [ ]:
# Load into LUSID
response = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=instruments,
    mapping_required={
        "name": "Name"
    },
    mapping_optional={},
    identifier_mapping={
        "Figi": "FIGI",
        "Isin": "ISIN",
        "ClientInternal": "ClientInternal",
        "Ticker": "Ticker"
    },
    property_columns=[
        "GICSSector", "GICSIndustry", "CountryCode", "SecurityType", "ExchangeCode", "Coupon", "PaymentFrequency", 
        "ParValue", "ConversionRatio", "ExerciseDate", "CounterParty", "ReplacementCost"
    ],
    file_type="instruments"
)

succ, failed, errors = format_instruments_response(response)
pd.DataFrame(data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}])

#### Market Data Quotes (Yesterday's Close)

In [ ]:
# Load and display quotes from CSV file
quotes = pd.read_csv("./data/IBORFlows_Quotes_27AprilClose.csv")

instruments_api = api_factory.build(InstrumentsApi)

# Resolve the quotes to the unique instrument in LUSID
quotes["LusidInstrumentId"] = quotes.apply(
    lambda x: instruments_api.get_instrument(identifier_type="Figi", identifier=x["FIGI"]).lusid_instrument_id 
              if not pd.isna(x["FIGI"]) else
              instruments_api.get_instrument(identifier_type="ClientInternal", identifier=x["ClientInternal"]).lusid_instrument_id
              if pd.isna(x["FXPair"]) else
              x["FXPair"],
    axis=1)

quotes["InstrumentIdType"] = quotes.apply(
    lambda x: "LusidInstrumentId" if pd.isna(x["FXPair"]) else "CurrencyPair",
    axis=1)

quotes.head(n=50)

In [ ]:
# Load the quotes into LUSID
response = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=quotes,
    mapping_required={
        "quote_id.effective_at": "$2020-04-27T20:00:00Z",
        "quote_id.quote_series_id.instrument_id_type": "InstrumentIdType",
        "quote_id.quote_series_id.provider": "$Lusid",
        "quote_id.quote_series_id.var_field": "$mid",
        "quote_id.quote_series_id.quote_type": "$Price",
        "quote_id.quote_series_id.instrument_id": "LusidInstrumentId",
        "metric_value.unit": "Currency",
        "metric_value.value": "Close Price"
    },
    mapping_optional={
        "lineage": "$YahooFinance"
    },
    file_type="quotes"
)

succ, failed, errors = format_quotes_response(response)
pd.DataFrame(data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}])

#### Create Portfolio

In [ ]:
transaction_portfolios_api = api_factory.build(TransactionPortfoliosApi)
corporate_actions_api = api_factory.build(CorporateActionSourcesApi)
corporate_action_code = "CorporateActionsStream"

try:
    # Create your Portfolio (in reality this would likely have been done already)
    response = transaction_portfolios_api.create_portfolio(
        scope=scope,
        create_transaction_portfolio_request=CreateTransactionPortfolioRequest(
            display_name="IBOR",
            description="Investment Book of Records",
            code=ibor_portfolio_code,
            created=ibor_portfolio_creation_date,
            base_currency="USD"
        )
    )

    print ("Porfolio Created")
    
except ApiException as e:
    print(json.loads(e.body)["title"])

try:
    # Create a Corporate Action Source to attach to the Portfolio
    response = corporate_actions_api.create_corporate_action_source(
        create_corporate_action_source_request=CreateCorporateActionSourceRequest(
            scope=scope,                                                             
            code=corporate_action_code, 
            display_name="StreamofCorporateActions", 
            description="Standard Stream of Corporate Actions")
    )
    
    print ("Corporate Action Source Created")
    
except ApiException as e:
    print(json.loads(e.body)["title"])

# Attach it to your Portfolio
transaction_portfolios_api.upsert_portfolio_details(
    scope=scope,
    code=ibor_portfolio_code,
    effective_at=ibor_portfolio_creation_date.isoformat(),
    create_portfolio_details=CreatePortfolioDetails(
        corporate_action_source_id=ResourceId(
            scope=scope,
            code=corporate_action_code)
    )
)

print ("Corporate Action Source Attached to Portfolio")

#### Set ABOR Holdings

In [ ]:
# Load ABOR positions from a file and display them
abor_day1 = pd.read_csv("./data/IBORFlows_Day1ABOR.csv")
abor_day1.head(n=50)

In [ ]:
# Load them into LUSID
response = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=abor_day1,
    mapping_required={
        "code": f"${ibor_portfolio_code}",
        "effective_at": f"${us_tz.localize(datetime(2020, 4, 28, 9)).astimezone(pytz.utc).isoformat()}",
        "tax_lots.units": "Quantity"
    },
    mapping_optional={
        "tax_lots.cost.amount": "Amount",
        "tax_lots.cost.currency": "Currency",
        "tax_lots.portfolio_cost": "PortfolioCost"
    },
    identifier_mapping={
        "Figi": "FIGI",
        "Currency": "InstrumentCurrency",
        "ClientInternal": "ClientInternal"
    },
    file_type="holdings"
)

succ, errors = format_holdings_response(response)
pd.DataFrame(data=[{"success": len(succ), "errors": len(errors)}])

In [ ]:
def get_holdings(effective_at_datetime, scope, portfolio_code):
    """
    Gets the holdings for a Portfolio at a given local datetime
    
    :parm: datetime effective_at_datetime: The datetime in the local timezone to get the holdings
    :parm: str scope: The scope of the Portfolio
    :parm: str code: The code of the Portfolio
    
    :returns: df A DataFrame with the holdings from the Portfolio
    """
    
    response = transaction_portfolios_api.get_holdings(
        scope=scope,
        code=portfolio_code,
        property_keys=[
            "Instrument/default/Name", 
            "Instrument/default/ClientInternal", 
            "Instrument/default/Figi"],
        effective_at=us_tz.localize(effective_at_datetime).astimezone(pytz.utc).isoformat())
    
    return lusid_response_to_data_frame(response, rename_properties=True)

get_holdings(datetime(2020, 4, 28, 9), scope, ibor_portfolio_code)

## 9:30am - Buy AAPL, Sell AMZN & GE Paid Dividends

- GE Paid Dividends (Corporate Action)
- Buy AAPL, Sell AMZN (Transactions)

#### GE Paid Dividends

In [ ]:
corporate_actions_api.batch_upsert_corporate_actions(
    scope=scope, 
    code="CorporateActionsStream", 
    upsert_corporate_action_request=[
        UpsertCorporateActionRequest(
            corporate_action_code="GEDividend",
            announcement_date=us_tz.localize(datetime(2020, 2, 28)).astimezone(pytz.utc).isoformat(),
            ex_date=us_tz.localize(datetime(2020, 4, 28, 9, 15)).astimezone(pytz.utc).isoformat(),
            record_date=us_tz.localize(datetime(2020, 4, 28, 9, 15)).astimezone(pytz.utc).isoformat(), 
            payment_date=us_tz.localize(datetime(2020, 4, 28, 9, 30)).astimezone(pytz.utc).isoformat(), 
            transitions=[
                CorporateActionTransitionRequest(
                    input_transition=CorporateActionTransitionComponentRequest(
                        instrument_identifiers={"Instrument/default/Figi" : "BBG000BK6MB5"}, 
                        units_factor=1, 
                        cost_factor=1), 
                    output_transitions=[CorporateActionTransitionComponentRequest(
                        instrument_identifiers={"Instrument/default/Currency" : "USD"}, 
                        units_factor=0.30, 
                        cost_factor=0)]
                )
            ]
        )
    ]
)

print ("Corporate Action Added")

#### Buy AAPL, Sell AMZN

In [ ]:
# Load transactions from a CSV file and display them
transactions = pd.read_csv("./data/IBORFlows_Transactions_930.csv")
transactions.head()

In [ ]:
# Load into LUSID
response = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=transactions,
    mapping_required={
        "code": f"${ibor_portfolio_code}",
        "transaction_id": "Id",
        "type": "TransactionType",
        "transaction_date": "$2020-04-28T13:30:00Z",
        "settlement_date": "$2020-04-30T13:30:00Z",
        "units": "Units",
        "total_consideration.amount": "Cost",
        "total_consideration.currency": "SettlementCurrency"
    },
    mapping_optional={
        "transaction_price.price": "Price",
        "transaction_price.type": "$Price",
        "transaction_currency": "TransactionCurrency"
    },
    identifier_mapping={
        "Ticker": "Ticker",
        "Figi": "FIGI",
    },
    property_columns=["Broker"],
    file_type="transactions"
)


succ, errors = format_transactions_response(response)
pd.DataFrame(data=[{"success": len(succ), "errors": len(errors)}])

In [ ]:
get_holdings(datetime(2020, 4, 28, 13, 30), scope, ibor_portfolio_code)

## 10:00am - New investor (cash injection)

In [ ]:
transactions = pd.read_csv("./data/IBORFlows_Transactions_1000.csv")
transactions.head()

In [ ]:
response = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=transactions,
    mapping_required={
        "code": f"${ibor_portfolio_code}",
        "transaction_id": "Id",
        "type": "TransactionType",
        "transaction_date": "$2020-04-28T14:00:00Z",
        "settlement_date": "$2020-04-29T14:00:00Z",
        "units": "Units",
        "total_consideration.amount": "Cost",
        "total_consideration.currency": "SettlementCurrency"
    },
    mapping_optional={
        "transaction_price.price": "Price",
        "transaction_price.type": "$Price",
        "transaction_currency": "TransactionCurrency"
    },
    identifier_mapping={
        "Currency": "TransactionCurrency"
    },
    property_columns=["InvestorId"],
    file_type="transactions"
)

succ, errors = format_transactions_response(response)
pd.DataFrame(data=[{"success": len(succ), "errors": len(errors)}])

In [ ]:
get_holdings(datetime(2020, 4, 28, 10), scope, ibor_portfolio_code)

## 11:00am - Convertible bond exercised into MSFT stock

- Configure "ExerciseConvertibleBond" transaction type 
- Load transaction

#### Create the properties for the transaction configurations

In [ ]:
properties_api = api_factory.build(PropertyDefinitionsApi)

def create_property_definition(domain, scope, code, data_type):
    try:
        properties_api.create_property_definition(
            create_property_definition_request=CreatePropertyDefinitionRequest(
                domain=domain,
                scope=scope,
                code=code,
                display_name=code,
                life_time="Perpetual",
                value_required=False,
                data_type_id=ResourceId(scope="system", code=data_type)
            )
        )
    except ApiException as e:
        detail = json.loads(e.body)
        if detail["code"] != 124:  # 'PropertyAlreadyExists'
            raise e
        else:
            print(f"property {domain}/{scope}/{code} already exists")
            
create_property_definition("Transaction", scope, "ResultingStockUnits", "number")
create_property_definition("Transaction", scope, "ResultingStockCost", "number")
create_property_definition("Transaction", scope, "ResultingStockId", "string")

#### Configure "ExerciseConvertibleBond" transaction type

In [ ]:
transaction_configuration_api = api_factory.build(TransactionConfigurationApi)

# Add default side definitions to the non-default transaction type scope.
# If working in the default scope, these side definitions are set by default so, unless these sides have been removed, this setting of sides can be skipped.
default_side_definitions = [
    SidesDefinitionRequest(
        side="Side1", 
        side_request=SideDefinitionRequest(
            security="Txn:LusidInstrumentId",
            currency="Txn:TradeCurrency",
            rate="Txn:TradeToPortfolioRate",
            units="Txn:Units",
            amount="Txn:TradeAmount")),
    SidesDefinitionRequest(
        side="Side2", 
        side_request=SideDefinitionRequest(
            security="Txn:SettleCcy",
            currency="Txn:SettlementCurrency",
            rate="SettledToPortfolioRate",
            units="Txn:TotalConsideration",
            amount="Txn:TotalConsideration"))
]

# Create a custom side config for your Exercise Transactions
exercise_side_definition = SidesDefinitionRequest(
        side="Exercise", 
        side_request=SideDefinitionRequest(
            security=f"Transaction/{scope}/ResultingStockId",
            currency="Txn:SettlementCurrency",
            rate="SettledToPortfolioRate",
            units=f"Transaction/{scope}/ResultingStockUnits",
            amount=f"Transaction/{scope}/ResultingStockCost"))

side_definitions = default_side_definitions[:]
side_definitions.append(exercise_side_definition)

transaction_configuration_api.set_side_definitions(side_definitions, scope = scope)

# Add default transaction types
default_transaction_mapping=open('data/default_transaction_mapping.json').read()
default_transaction_mapping = json.loads(default_transaction_mapping)

def map_properties(properties):
    return {property["key"]: PerpetualProperty(key=property["key"], value=PropertyValue(label_value=property["value"])) for property in properties}
def map_alias(alias):
    return TransactionTypeAlias(type=alias["type"], description=alias["description"], transaction_class=alias["transactionClass"], transaction_roles=alias["transactionRoles"])
def map_movement(movement):
    return TransactionTypeMovement(movement_types=movement["movementTypes"], side=movement["side"], direction=movement["direction"], properties=map_properties(movement["properties"]))
def map_transaction_type_request(transaction_type_request):
    return TransactionTypeRequest(
        aliases=[map_alias(alias) for alias in transaction_type_request["aliases"]],
        movements=[map_movement(movement) for movement in transaction_type_request["movements"]],
        properties=map_properties(transaction_type_request["properties"]))

for configuration in default_transaction_mapping:
    transaction_type_requests = [map_transaction_type_request(transaction_type_request) for transaction_type_request in configuration["transactionTypeRequests"]]
    
    # Call LUSID to set your configuration for our transaction types
    transaction_configuration_api.set_transaction_type_source(
        source=configuration["source"],
        transaction_type_request=transaction_type_requests,
        scope=scope
    )

# Add a transaction type for "ExerciseConvertibleBond" using this side

exercise_movements = [
    TransactionTypeMovement(
        movement_types='StockMovement',
        side='Side1',
        direction=-1,
        properties=None,
        mappings=None),
    TransactionTypeMovement(
        movement_types='StockMovement',
        side='Exercise',
        direction=1,
        properties=None,
        mappings=None)
]

response = transaction_configuration_api.set_transaction_type(
    source="ExerciseTransaction",
    type="ExerciseConvertibleBond",
    transaction_type_request=TransactionTypeRequest(
        aliases=[
            TransactionTypeAlias(
                type="ExerciseConvertibleBond",
                description="The exercise of a convertible bond into stocks",
                transaction_class="ExerciseTransactions",
                transaction_roles="Longer",
            )
        ],
        movements=exercise_movements,
    ),
    scope=scope
)

print ("Created side Transaction Types ExerciseConvertibleBond")


# Call LUSID to update the transaction type scope of your portfolio 
patch_document = [
    {
        "value": scope,
        "path": "/transactiontypescope",
        "op": "add"
    }
]
patch_response = api_factory.build(TransactionPortfoliosApi).patch_portfolio_details(
    scope=scope,
    code=ibor_portfolio_code,
    operation=patch_document)

#### Load transaction

In [ ]:
transactions = pd.read_csv("./data/IBORFlows_Transactions_1100.csv")

# Resolve external FIGI identifier of resulting stock to internal LUSID Instrument Id identifier
transactions["ResultingStockId"] = transactions.apply(lambda x: instruments_api.get_instrument(
    identifier_type="Figi", identifier=x["ResultingStockFigi"]).lusid_instrument_id, axis=1)

transactions.head()

In [ ]:
response = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=transactions,
    mapping_required={
        "code": f"${ibor_portfolio_code}",
        "transaction_id": "Id",
        "type": "TransactionType",
        "transaction_date": "$2020-04-28T15:00:00Z",
        "settlement_date": "$2020-04-30T15:00:00Z",
        "units": "Units",
        "total_consideration.amount": "Cost",
        "total_consideration.currency": "SettlementCurrency"
    },
    mapping_optional={
        "transaction_price.price": "Price",
        "transaction_price.type": "$Price",
        "transaction_currency": "TransactionCurrency",
        "source": "$ExerciseTransaction"
    },
    identifier_mapping={
        "ClientInternal": "ClientInternal"
    },
    property_columns=[
        "ResultingStockFigi", 
        "ResultingStockId", 
        "ResultingStockTicker", 
        "ResultingStockUnits", 
        "ResultingStockCost"
    ],
    file_type="transactions"
)

succ, errors = format_transactions_response(response)
pd.DataFrame(data=[{"success": len(succ), "errors": len(errors)}])

In [ ]:
get_holdings(datetime(2020, 4, 28, 11), scope, ibor_portfolio_code)

## 12:00pm - Rights exercised into FB stock

- Configure "ExerciseRights" transaction type
- Load transaction

#### Configure "ExerciseRights" transaction type

In [ ]:
# Configure the ExerciseRights transaction type using the same movements as earlier
response = transaction_configuration_api.set_transaction_type(
    source="ExerciseTransaction",
    type="ExerciseRights",
    transaction_type_request=TransactionTypeRequest(
        aliases=[
            TransactionTypeAlias(
                type="ExerciseRights",
                description="The exercise of a rights issuance into stocksd",
                transaction_class="ExerciseTransactions",
                transaction_roles="Longer",
            )
        ],
        movements=exercise_movements,
    ),
    scope = scope
)

#### Load transaction

In [ ]:
transactions = pd.read_csv("./data/IBORFlows_Transactions_1200.csv")

transactions["ResultingStockId"] = transactions.apply(lambda x: instruments_api.get_instrument(
    identifier_type="Figi", identifier=x["ResultingStockFigi"]).lusid_instrument_id, axis=1)

transactions.head()

In [ ]:
response = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=transactions,
    mapping_required={
        "code": f"${ibor_portfolio_code}",
        "transaction_id": "Id",
        "type": "TransactionType",
        "transaction_date": "$2020-04-28T16:00:00Z",
        "settlement_date": "$2020-04-30T16:00:00Z",
        "units": "Units",
        "total_consideration.amount": "Cost",
        "total_consideration.currency": "SettlementCurrency"
    },
    mapping_optional={
        "transaction_price.price": "Price",
        "transaction_price.type": "$Price",
        "transaction_currency": "TransactionCurrency",
        "source": "$ExerciseTransaction"
    },
    identifier_mapping={
        "ClientInternal": "ClientInternal"
    },
    property_columns=[
        "ResultingStockFigi", 
        "ResultingStockId", 
        "ResultingStockTicker", 
        "ResultingStockUnits", 
        "ResultingStockCost"
    ],
    file_type="transactions"
)

succ, errors = format_transactions_response(response)
pd.DataFrame(data=[{"success": len(succ), "errors": len(errors)}])

In [ ]:
get_holdings(datetime(2020, 4, 28, 12), scope, ibor_portfolio_code)

## 1:00pm - Do I have enough cash to buy NFLX? Will I overdraft? What about in 3 days time when it settles?

- Check cash balance now
- Check cash balance in 3 days time

In [ ]:
# Load trade you are considering from CSV file
transactions = pd.read_csv("./data/IBORFlows_Transactions_1300.csv")
transaction_cost = transactions.iloc[0]["Cost"]
transactions.head()

#### Check cash balance now

In [ ]:
response = transaction_portfolios_api.get_holdings(
    scope=scope,
    code=ibor_portfolio_code,
    effective_at=us_tz.localize(datetime(2020, 4, 28, 13)).astimezone(pytz.UTC).isoformat()
)

available_cash = sum([holding.units for holding in response.values if 
                      holding.instrument_uid == "CCY_USD" and
                      holding.holding_type == "B"])

print("USD${:,.2f}".format(available_cash), "in available funds")
print("USD${:,.2f}".format(transaction_cost), "to make NFLX transaction")
print("USD${:,.2f}".format(available_cash-transaction_cost), "remaining after transaction")

#### Check settled cash balance in 3 days

In [ ]:
response = transaction_portfolios_api.get_holdings(
    scope=scope,
    code=ibor_portfolio_code,
    effective_at=us_tz.localize(datetime(2020, 5, 1, 13)).astimezone(pytz.UTC).isoformat()
)

available_cash = sum([holding.units for holding in response.values if 
                      holding.instrument_uid == "CCY_USD" and
                      holding.holding_type == "B"])

print("USD${:,.2f}".format(available_cash), "in available funds")
print("USD${:,.2f}".format(transaction_cost), "to make NFLX transaction")
print("USD${:,.2f}".format(available_cash-transaction_cost), "remaining after transaction")

## Will it breach my allowable Tech or US exposure?

- Check current exposure by GICS Sector
- Check current exposure by Country
- Simulate trade
- Check simulated exposure by GICS Sector
- Check simulated exposure by Country

#### Current Exposure by GICS Sector

You are not permitted to have more than 30% in Information Technology.

In [ ]:
aggregation_api = api_factory.build(AggregationApi)

def chart_exposure(aggregation_api, effective_at, exposure_by, interest, scope, portfolio_code):
    """
    This function retrieves and charts the exposure of the Portofolio
    
    :param: aggreagation_api: The LUSID aggregation API to use
    :param: effective_at: The effective at date to chart the exposure
    :param: exposure_by: The metric to chart exposure by e.g. GICSSector
    :param: interest: The area of interest e.g. Information Technology, this will explode on the chart
    :param: scope: The scope of the Portfolio to look at exposure for
    :param: portfolio_code: The code of the Portfolio to look at exposure for
    """    
    # Create the valuation request
    valuation_request = ValuationRequest(
        recipe_id= ResourceId(scope=scope, code='default'),
        metrics=[
            AggregateSpec(key='Valuation/PvInReportCcy',
            op='Sum'),
            AggregateSpec(key=f'Instrument/{scope}/{exposure_by}',
            op='Value')
        ],
        group_by=[
            f'Instrument/{scope}/{exposure_by}'
        ],
        portfolio_entity_ids=[PortfolioEntityId(scope=scope, code=portfolio_code)],
        valuation_schedule=ValuationSchedule(effective_at=us_tz.localize(effective_at).astimezone(pytz.UTC).isoformat())
    )

    # Perform a valuation
    response = api_factory.build(AggregationApi).get_valuation(
        valuation_request=valuation_request)

    labels = [result[f'Instrument/{scope}/{exposure_by}'] for result in response.data]
    values = [result['Sum(Valuation/PvInReportCcy)'] for result in response.data]
    
    explode = [0] * len(labels)
    index = labels.index(interest)
    explode[index] = 0.1

    fig1, ax1 = plt.subplots(figsize=(10,10))
    ax1.pie(values, explode=explode, labels=labels, autopct='%1.1f%%',
            shadow=True, startangle=90)
    ax1.axis('equal')

    plt.show()
    
chart_exposure(aggregation_api, datetime(2020, 4, 28, 13), "GICSSector", "Information Technology", scope, ibor_portfolio_code)

#### Current Exposure by Country

You are not permitted to have more than 60% exposure to the US.

In [ ]:
chart_exposure(
    aggregation_api, 
    datetime(2020, 4, 28, 13), 
    "CountryCode", 
    "US", 
    scope, 
    ibor_portfolio_code
)

#### Simulate the Transaction

In [ ]:
derived_transaction_portfolios_api = api_factory.build(DerivedTransactionPortfoliosApi)
nflx_simulated_ibor_portfolio_code = "IBOR-NFLX-Trade-Simulation"


try: 
    derived_transaction_portfolios_api.create_derived_portfolio(
        scope=scope,
        create_derived_transaction_portfolio_request=CreateDerivedTransactionPortfolioRequest(
            display_name="NFLXTradeSimulation",
            created=ibor_portfolio_creation_date,
            code=nflx_simulated_ibor_portfolio_code,
            parent_portfolio_id=ResourceId(
                scope=scope,
                code=ibor_portfolio_code)
        )
    )
    
except ApiException as e:
    print(json.loads(e.body)["title"])

response = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=transactions,
    mapping_required={
        "code": f"${nflx_simulated_ibor_portfolio_code}",
        "transaction_id": "Id",
        "type": "TransactionType",
        "transaction_date": "$2020-04-28T17:00:00Z",
        "settlement_date": "$2020-05-01T17:00:00Z",
        "units": "Units",
        "total_consideration.amount": "Cost",
        "total_consideration.currency": "SettlementCurrency"
    },
    mapping_optional={
        "transaction_price.price": "Price",
        "transaction_price.type": "$Price",
        "transaction_currency": "TransactionCurrency"
    },
    identifier_mapping={
        "Ticker": "Ticker",
        "Figi": "Figi",
    },
    file_type="transactions"
)

succ, errors = format_transactions_response(response)
pd.DataFrame(data=[{"success": len(succ), "errors": len(errors)}])

#### Simulated Exposure by GICS Sector

In [ ]:
chart_exposure(
    aggregation_api, 
    datetime(2020, 4, 28, 14), 
    "GICSSector", 
    "Information Technology", 
    scope, 
    nflx_simulated_ibor_portfolio_code
)

#### Simulated Exposure by Country

In [ ]:
chart_exposure(
    aggregation_api, 
    datetime(2020, 4, 28, 14), 
    "CountryCode", 
    "US", 
    scope, 
    nflx_simulated_ibor_portfolio_code
)

#### Make the Trade for Real

In [ ]:
response = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=transactions,
    mapping_required={
        "code": f"${ibor_portfolio_code}",
        "transaction_id": "Id",
        "type": "TransactionType",
        "transaction_date": "$2020-04-28T15:00:00Z",
        "settlement_date": "$2020-04-30T15:00:00Z",
        "units": "Units",
        "total_consideration.amount": "Cost",
        "total_consideration.currency": "SettlementCurrency"
    },
    mapping_optional={
        "transaction_price.price": "Price",
        "transaction_price.type": "$Price",
        "transaction_currency": "TransactionCurrency"
    },
    identifier_mapping={
        "Ticker": "Ticker",
        "Figi": "Figi",
    },
    file_type="transactions"
)

succ, errors = format_transactions_response(response)
pd.DataFrame(data=[{"success": len(succ), "errors": len(errors)}])

In [ ]:
get_holdings(datetime(2020, 4, 28, 11), scope, ibor_portfolio_code)

## 2:00pm - I want to enter into an IRS with a given counterparty, What is my counterparty exposure right now? What is the risk level of this counterparty?

Upsert IRS security

How to calculate exposure?
What is risk level in this context?

In [ ]:
# Get holdings with counteryparty related properties 
response = transaction_portfolios_api.get_holdings(
    scope=scope,
    code=ibor_portfolio_code,
    effective_at=us_tz.localize(datetime(2020, 4, 28, 13)).astimezone(pytz.UTC).isoformat(),
    property_keys=[
        f"Instrument/{scope}/SecurityType",
        f"Instrument/{scope}/CounterParty",
        f"Instrument/{scope}/ReplacementCost"
    ]
)

# Filter to relevant over the counter (OTC) contracts
otc_holdings = [holding for holding in response.values if holding.properties.get(
    f'Instrument/{scope}/SecurityType', None) is not None]

otc_holdings = [holding for holding in otc_holdings if holding.properties.get(
    f'Instrument/{scope}/SecurityType', None).value.label_value == "Interest Rate Swap" ]

# Determine counterparty exposure
counterparty_exposure = {}

for holding in otc_holdings:
    counterparty_exposure.setdefault(holding.properties[f"Instrument/{scope}/CounterParty"].value.label_value, []).append(
        holding.properties[f"Instrument/{scope}/ReplacementCost"].value.metric_value.value)
    
counterparty_exposure = {counterparty: sum(replacement_costs) for counterparty, replacement_costs in counterparty_exposure.items()}

# Chart results
y_pos = np.arange(len(counterparty_exposure.keys()))
plt.bar(y_pos, list(counterparty_exposure.values()), align='center', alpha=0.5)
plt.xticks(y_pos, list(counterparty_exposure.keys()))
plt.ylabel('Total Net Replacement Cost')
plt.title('Counterparty Exposure')
plt.show()

## 4:15pm - Buy more convertible MSFT debt

Buy MSFT convertible bond

In [ ]:
transactions = pd.read_csv("./data/IBORFlows_Transactions_1615.csv")
transactions.head()

In [ ]:
response = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=transactions,
    mapping_required={
        "code": f"${ibor_portfolio_code}",
        "transaction_id": "Id",
        "type": "TransactionType",
        "transaction_date": "$2020-04-28T20:15:00Z",
        "settlement_date": "$2020-04-30T20:15:00Z",
        "units": "Units",
        "total_consideration.amount": "Cost",
        "total_consideration.currency": "SettlementCurrency"
    },
    mapping_optional={
        "transaction_price.price": "Price",
        "transaction_price.type": "$Price",
        "transaction_currency": "TransactionCurrency",
    },
    identifier_mapping={
        "ClientInternal": "ClientInternal"
    },
    file_type="transactions"
)

succ, errors = format_transactions_response(response)
pd.DataFrame(data=[{"success": len(succ), "errors": len(errors)}])

In [ ]:
get_holdings(datetime(2020, 4, 28, 16, 15), scope, ibor_portfolio_code)

## EOD Trading - How much committed unsettled cash do I have? For how long? I want to park that in a money market account.

- Get Holdings & produce cash ladder

In [ ]:
response = transaction_portfolios_api.get_holdings(
    scope=scope,
    code=ibor_portfolio_code,
    effective_at=us_tz.localize(datetime(2020, 4, 28, 17)).astimezone(pytz.UTC).isoformat(),
)

cash_positions = [holding for holding in response.values if "CCY_" in holding.instrument_uid and holding.holding_type != "B"]

settle_amounts = {}

for holding in cash_positions:
    settle_amounts.setdefault(f"{holding.cost.currency}@{holding.transaction.settlement_date}", []).append(
        holding.cost.amount)

settle_amounts = {key: sum(value) for key, value in settle_amounts.items()} 
settle_amounts


x = [datetime.strptime(key.split("@")[1], "%Y-%m-%d %H:%M:%S%z").astimezone(us_tz) for key in list(settle_amounts.keys())]
y = [value for value in list(settle_amounts.values())]

# Chart results
fig, ax = plt.subplots(figsize=(20, 8))
y_pos = np.arange(len(x))
ax.bar(y_pos, y, align='center', alpha=0.5)
plt.ylim(min(y)-abs(min(y)/4), max(y)+abs(min(y))/4)
plt.xticks(y_pos, x)
plt.gcf().axes[0].yaxis.get_major_formatter().set_scientific(False)
plt.ylabel('USD')
plt.xlabel('Settlement Date')
plt.title('Cash')
plt.show()

## Day 2

## 8am - Start of day holdings from ABOR, reconcile with IBOR from yesterday at NAV level, NAV is off

Will ensure that you have the following loaded internally to support the start of day ABOR

- Market Data Quotes

Then you load in the start of day ABOR

- Upsert NAV to Results Store
- Conduct NAV reconciliation

#### Market Data Quotes

In [ ]:
quotes = pd.read_csv("./data/IBORFlows_Quotes_28AprilClose.csv")

instruments_api = api_factory.build(InstrumentsApi)

# Resolve the quotes to the instrument in LUSID using the Lusid Instrument Id
quotes["LusidInstrumentId"] = quotes.apply(
    lambda x: instruments_api.get_instrument(identifier_type="Figi", identifier=x["FIGI"]).lusid_instrument_id 
              if not pd.isna(x["FIGI"]) else
              instruments_api.get_instrument(identifier_type="ClientInternal", identifier=x["ClientInternal"]).lusid_instrument_id
              if pd.isna(x["FXPair"]) else
              x["FXPair"],
    axis=1)

quotes["InstrumentIdType"] = quotes.apply(
    lambda x: "LusidInstrumentId" if pd.isna(x["FXPair"]) else "CurrencyPair",
    axis=1)

quotes.head(n=50)

In [ ]:
response = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=quotes,
    mapping_required={
        "quote_id.effective_at": "$2020-04-28T20:00:00Z",
        "quote_id.quote_series_id.instrument_id_type": "InstrumentIdType",
        "quote_id.quote_series_id.provider": "$Lusid",
        "quote_id.quote_series_id.var_field": "$mid",
        "quote_id.quote_series_id.quote_type": "$Price",
        "quote_id.quote_series_id.instrument_id": "LusidInstrumentId",
        "metric_value.unit": "Currency",
        "metric_value.value": "Close Price"
    },
    mapping_optional={
        "lineage": "$YahooFinance"
    },
    file_type="quotes"
)

succ, failed, errors = format_quotes_response(response)
pd.DataFrame(data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}])

#### Upsert NAV to Results Store

In [ ]:
abor_day2 = pd.read_csv("./data/IBORFlows_Day2ABORWithNAV.csv")
abor_day2.head(n=5)

In [ ]:
results_api = api_factory.build(StructuredResultDataApi)

data_map_key = DataMapKey(
    code = "sample-data-map",
    version = "1.0.2"
)

try:
    results_api.create_data_map(
        scope = scope,
        request_body = {
            "data-map": CreateDataMapRequest(
                id=data_map_key,
                data=DataMapping(data_definitions=[
                    DataDefinition(address="Instrument/default/LusidInstrumentId", data_type="string", name="LusidInstrumentId", key_type="PartOfUnique"),
                    DataDefinition(address="Holding/default/Currency", data_type="string", name="Currency", key_type="PartOfUnique"),
                    DataDefinition(address="Valuation/PvInReportCcy", data_type="Result0D", key_type="CompositeLeaf"),
                    DataDefinition(address="Valuation/PvInReportCcy/Amount", data_type="decimal", key_type="Leaf", name="NAV"),
                    DataDefinition(address="Valuation/PvInReportCcy/Ccy", data_type="string", key_type="Leaf", name="ValuationCurrency"),
                    DataDefinition(address="Instrument/default/Name", data_type="string", key_type="Leaf", name="InstrumentName"),
                    DataDefinition(address="Holding/default/PortfolioCost", data_type="decimal", key_type="Leaf", name="PortfolioCost"),
                    DataDefinition(address="Holding/default/Units", data_type="decimal", key_type="Leaf", name="Units"),
                ])
            )
        }
    )
except:
    print("DataMaps are immutable - a datamap under this key already exists")
    
# Resolve each instrument to the unique instrument in LUSID
abor_day2["LusidInstrumentId"] = abor_day2.apply(
    lambda x: instruments_api.get_instrument(identifier_type="Figi", identifier=x["Figi"]).lusid_instrument_id 
              if not pd.isna(x["Figi"]) else
              instruments_api.get_instrument(identifier_type="ClientInternal", identifier=x["ClientInternal"]).lusid_instrument_id
              if not pd.isna(x["ClientInternal"]) else
              f"CCY_{x['Currency']}",
    axis=1)

# Save then read the file as a CSV
abor_day2.to_csv("./data/IBORFlows_Day2ABORWithNAVLUIDS.csv", index=False)
with open("./data/IBORFlows_Day2ABORWithNAVLUIDS.csv", 'r') as myfile: 
    fdata = myfile.read()

# Upsert the results into LUSID
result_id = StructuredResultDataId(
    source = "Client",
    code = ibor_portfolio_code + "-results",
    effective_at = us_tz.localize(datetime(2020, 4, 29, 8)).astimezone(pytz.utc).isoformat(), 
    result_type = "UnitResult/Portfolio"
)

result_data = StructuredResultData(
    document_format = "Csv", 
    version = "1.0.0", 
    name = ibor_portfolio_code + "-results", 
    document = fdata, 
    data_map_key = data_map_key
)

results_api.upsert_structured_result_data(
    scope = scope,
    request_body = {
        "data": UpsertStructuredResultDataRequest(id=result_id, data=result_data) 
    }
)

print ("Loaded ABOR NAV data into LUSID")

# create a recipe for lookig up the the result
api_factory.build(ConfigurationRecipeApi).upsert_configuration_recipe(
    UpsertRecipeRequest(
        configuration_recipe=ConfigurationRecipe(
        scope=scope,
        code="lookup-results",
        pricing=PricingContext(
            options={
                "AllowPartiallySuccessfulEvaluation": True
            },
            result_data_rules=[
                PortfolioResultDataKeyRule(
                    supplier="Client",
                    data_scope=scope,
                    document_code=ibor_portfolio_code + "-results",
                    result_key_rule_type="PortfolioResultDataKeyRule"
                )]
            )
        )
    )
)
print("Created result lookup recipe")


#### Conduct NAV reconciliation

In [ ]:
reconciliation_api = api_factory.build(ReconciliationsApi)

# Aggregation request for IBOR
lhs_valuation_request = ValuationRequest(
    recipe_id= ResourceId(scope=scope, code='default'),
    metrics=[
        AggregateSpec(key='Valuation/PvInReportCcy',
        op='Sum'),
        AggregateSpec(key='Holding/default/Units',
        op='Sum'),
        AggregateSpec(key='Instrument/default/LusidInstrumentId',
        op='Value'),
        AggregateSpec(key='Valuation/PvInReportCcy/Ccy',
        op='Value'),
    ],
    group_by=[
        'Valuation/PvInReportCcy/Ccy',
        'Instrument/default/LusidInstrumentId',
    ],
    report_currency="USD",
    portfolio_entity_ids=[PortfolioEntityId(scope=scope, code=ibor_portfolio_code)],
    valuation_schedule=ValuationSchedule(effective_at=us_tz.localize(datetime(2020, 4, 29, 8)).astimezone(pytz.UTC).isoformat())
    )

rhs_valuation_request = ValuationRequest(
    recipe_id= ResourceId(scope=scope, code='lookup-results'),
    metrics=[
        AggregateSpec(key='Valuation/PvInReportCcy',
        op='Sum'),
        AggregateSpec(key='Holding/default/Units',
        op='Sum'),
        AggregateSpec(key='Instrument/default/LusidInstrumentId',
        op='Value'),
        AggregateSpec(key='Valuation/PvInReportCcy/Ccy',
        op='Value'),
    ],
    group_by=[
        'Valuation/PvInReportCcy/Ccy',
        'Instrument/default/LusidInstrumentId'
    ],
    report_currency="USD",
    portfolio_entity_ids=[PortfolioEntityId(scope=scope, code=ibor_portfolio_code)],
    valuation_schedule=ValuationSchedule(effective_at=us_tz.localize(datetime(2020, 4, 29, 8)).astimezone(pytz.utc).isoformat())
    )

# Run reconciliation
response = reconciliation_api.reconcile_valuation(
    valuations_reconciliation_request=ValuationsReconciliationRequest(
        left=lhs_valuation_request,
        right=rhs_valuation_request,
        preserve_keys=[
            'Instrument/default/LusidInstrumentId',
        ]
))

left = pd.DataFrame(response.left.data).rename(columns={
    "Sum(Holding/default/Units)": "IBOR_units",
    "Sum(Valuation/PvInReportCcy)": "IBOR_NAV.amount",
    "Instrument/default/LusidInstrumentId": "instrument_uid",
    "Valuation/PvInReportCcy/Ccy": "pvccy_lhs"
})
right = pd.DataFrame(response.right.data).rename(columns={
    "Sum(Holding/default/Units)": "ABOR_units",
    "Sum(Valuation/PvInReportCcy)": "ABOR_NAV.amount",
    "Instrument/default/LusidInstrumentId": "instrument_uid",
    "Valuation/PvInReportCcy/Ccy": "pvccy_rhs"
})
diff = pd.DataFrame(response.diff).rename(columns={
    "Sum(Holding/default/Units)": "difference_units",
    "Sum(Valuation/PvInReportCcy)": "difference_NAV.amount",
    "Instrument/default/LusidInstrumentId": "instrument_uid",
    "Valuation/PvInReportCcy/Ccy": "pvccy"
})


In [ ]:
df = left.join(right.set_index("instrument_uid"), on="instrument_uid") \
        .join(diff.set_index("instrument_uid"), on="instrument_uid")
df = df[df["difference_units"] != 0]
df["instrument_name"] = df.apply(
    lambda x: instruments_api.get_instrument(
                identifier_type="LusidInstrumentId", identifier=x["instrument_uid"]).name
                if not "CCY" in x["instrument_uid"] else
                f"Cash {x['instrument_uid'].split('_')[1]}",                  
    axis=1)

df.head(n=20)


## Are all positions showing up? Is cash lining up?

New injections/withdrawls confirmed? Cash from corporate actions/options correct?

In [ ]:
response = transaction_portfolios_api.get_holdings(
    scope=scope,
    code=ibor_portfolio_code,
    effective_at=us_tz.localize(datetime(2020, 4, 29, 8)).astimezone(pytz.UTC).isoformat()
)

settled_cash = sum([holding.units for holding in response.values if 
                      holding.instrument_uid == "CCY_USD" and
                      holding.holding_type == "B"])

unsettled_cash = sum([holding.units for holding in response.values if 
                      holding.instrument_uid == "CCY_USD" and
                      holding.holding_type != "B"])

print("USD${:,.2f}".format(settled_cash), "in settled cash")
print("USD${:,.2f}".format(unsettled_cash), "in unsettled cash")
print("USD${:,.2f}".format(settled_cash+unsettled_cash), "in total cash")

In [ ]:
unsettled_cash = [holding for holding in response.values if 
                      holding.instrument_uid == "CCY_USD" and
                      holding.holding_type != "B"]

lusid_response_to_data_frame(unsettled_cash, rename_properties=True)

#### Tear Down

In [ ]:
portfolios_api = api_factory.build(PortfoliosApi)
corporate_actions_api = api_factory.build(CorporateActionSourcesApi)
nflx_simulated_ibor_portfolio_code = "IBOR-NFLX-Trade-Simulation"
portfolios_api.delete_portfolio(scope=scope, code=nflx_simulated_ibor_portfolio_code)
corporate_actions_api.delete_corporate_action_source(scope=scope, code="CorporateActionsStream")
portfolios_api.delete_portfolio(scope=scope, code=ibor_portfolio_code)
print ("Tear down complete")